In [1]:
!nvidia-smi

Thu Nov 13 16:19:31 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce MX330           Off |   00000000:2C:00.0 Off |                  N/A |
| N/A   51C    P8            N/A  / 5001W |       7MiB /   2048MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))

1.13.1+cu117
11.7
NVIDIA GeForce MX330


In [4]:
import os
import cv2
import math
import time
import random
import pathlib
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt

In [10]:
import cv2
import threading
import queue
from ultralytics import YOLO


class PersonStreamDetector(threading.Thread):
    """
    Threaded YOLOv8-based person detector for video streams.
    Keeps only the latest frame result in a single-element queue.
    """

    def __init__(self, source=0, model_path="yolov8l.pt", conf_thres=0.5, output_queue=None, bbox_expand=1.1):
        super().__init__()
        self.source = source
        self.model_path = model_path
        self.conf_thres = conf_thres
        self.output_queue = output_queue or queue.Queue(maxsize=1)
        self.bbox_expand = bbox_expand

        self.model = YOLO(self.model_path)
        self.cap = cv2.VideoCapture(self.source)

        if not self.cap.isOpened():
            raise RuntimeError(f"❌ Unable to open video source: {self.source}")

        self.running = True

    def run(self):
        print("🎥 YOLO person detection thread started.")

        while self.running:
            ret, frame = self.cap.read()
            if not ret:
                continue  # skip empty frames, do not break (e.g. webcam hiccups)

            results = self.model(frame, conf=self.conf_thres, verbose=False)

            detections = []
            for box in results[0].boxes:
                if int(box.cls[0]) != 0:  # class 0 = person
                    continue

                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])

                # Expand bbox
                w, h = x2 - x1, y2 - y1
                dw, dh = int((self.bbox_expand - 1) * w / 2), int((self.bbox_expand - 1) * h / 2)
                x1e = max(0, x1 - dw)
                y1e = max(0, y1 - dh)
                x2e = min(frame.shape[1], x2 + dw)
                y2e = min(frame.shape[0], y2 + dh)

                crop = frame[y1e:y2e, x1e:x2e]
                detections.append({
                    "bbox": (x1e, y1e, x2e, y2e),
                    "confidence": conf,
                    "crop": crop
                })

            # Replace the existing queue item (latest frame only)
            if not self.output_queue.empty():
                try:
                    self.output_queue.get_nowait()
                except queue.Empty:
                    pass

            self.output_queue.put_nowait({
                "frame": frame,
                "detections": detections
            })

        self.cap.release()
        print("🛑 YOLO detection thread stopped.")

    def stop(self):
        self.running = False

In [18]:
import cv2
import numpy as np
import queue
from collections import deque
from datetime import datetime
from ultralytics import YOLO
import os

# === Configuration ===
GRID_ROWS, GRID_COLS = 4, 4
IMG_SIZE = 100
SAVE_FALLS = True
SAVE_DIR = "detected_falls"
WINDOW_MAIN_SIZE = (960, 540)
WINDOW_GRID_SIZE = (800, 800)
WINDOW_LAST_FALL_SIZE = (800, 800)

# === Prepare save directory ===
if SAVE_FALLS:
    os.makedirs(SAVE_DIR, exist_ok=True)

# === Load classifier model ===
classifier = YOLO("YOLO11N/train7/weights/best.pt")

# === Initialize detector ===
output_q = queue.Queue(maxsize=1)
detector = PersonStreamDetector(source=0, output_queue=output_q, bbox_expand=1.3)
detector.start()

crop_history = deque(maxlen=15)
last_fall_img = None  # store the most recent fall grid image


def draw_prediction_label(img, nofall_conf, fall_conf, is_fall, origin=(20, 40)):
    """Draw FALL / NO FALL text with confidence and highlight."""
    font = cv2.FONT_HERSHEY_SIMPLEX
    highlight_color = (0, 255, 255)  # Yellow
    normal_color = (180, 180, 180)
    text_scale = 0.8
    thickness = 2
    x, y = origin
    line_height = 40

    # NO FALL
    cv2.putText(
        img,
        f"NO FALL: {nofall_conf:.2f}",
        (x, y),
        font,
        text_scale,
        highlight_color if not is_fall else normal_color,
        thickness,
    )
    # FALL
    cv2.putText(
        img,
        f"FALL: {fall_conf:.2f}",
        (x, y + line_height),
        font,
        text_scale,
        highlight_color if is_fall else normal_color,
        thickness,
    )


while True:
    if not output_q.empty():
        data = output_q.get()
        frame = data["frame"]

        # --- Collect and store last 15 person crops ---
        for det in data["detections"]:
            x1, y1, x2, y2 = det["bbox"]
            crop = det["crop"]
            resized = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))
            crop_history.append(resized)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # --- Build grid image ---
        if len(crop_history) > 0:
            grid_img = np.zeros((GRID_ROWS * IMG_SIZE, GRID_COLS * IMG_SIZE, 3), dtype=np.uint8)
            for i, img in enumerate(crop_history):
                row = i // GRID_COLS
                col = i % GRID_COLS
                y1, y2 = row * IMG_SIZE, (row + 1) * IMG_SIZE
                x1, x2 = col * IMG_SIZE, (col + 1) * IMG_SIZE
                grid_img[y1:y2, x1:x2] = img

            # --- Perform classification ---
            results = classifier.predict(grid_img, verbose=False)
            probs = results[0].probs.data.cpu().numpy()

            nofall_conf = float(probs[0])
            fall_conf = float(probs[1])
            is_fall = fall_conf > nofall_conf

            # === Draw predictions on BOTH windows ===
            draw_prediction_label(frame, nofall_conf, fall_conf, is_fall, origin=(20, 40))
            draw_prediction_label(grid_img, nofall_conf, fall_conf, is_fall, origin=(20, 40))

            # --- Save if fall detected ---
            if is_fall:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                if SAVE_FALLS:
                    path = os.path.join(SAVE_DIR, f"fall_{timestamp}_conf{fall_conf:.2f}.jpg")
                    cv2.imwrite(path, grid_img)
                    print(f"[SAVED] Fall detected → {path}")
                last_fall_img = grid_img.copy()

            # --- Show grid image ---
            cv2.namedWindow("Person Crop Grid", cv2.WINDOW_NORMAL)
            cv2.resizeWindow("Person Crop Grid", *WINDOW_GRID_SIZE)
            cv2.imshow("Person Crop Grid", grid_img)

        # --- Show main detector stream ---
        cv2.namedWindow("YOLO Stream Detector", cv2.WINDOW_NORMAL)
        cv2.resizeWindow("YOLO Stream Detector", *WINDOW_MAIN_SIZE)
        cv2.imshow("YOLO Stream Detector", frame)

        # --- Show last fall image if available ---
        if last_fall_img is not None:
            cv2.namedWindow("Last Fall Detected", cv2.WINDOW_NORMAL)
            cv2.resizeWindow("Last Fall Detected", *WINDOW_LAST_FALL_SIZE)
            cv2.imshow("Last Fall Detected", last_fall_img)

    # --- Stop condition ---
    if cv2.waitKey(1) & 0xFF == ord('q'):
        detector.stop()
        detector.join()
        break

cv2.destroyAllWindows()

🎥 YOLO person detection thread started.
[SAVED] Fall detected → detected_falls/fall_20251113_165915_conf0.66.jpg
[SAVED] Fall detected → detected_falls/fall_20251113_165918_conf0.53.jpg
[SAVED] Fall detected → detected_falls/fall_20251113_165919_conf0.52.jpg
[SAVED] Fall detected → detected_falls/fall_20251113_165924_conf0.91.jpg
[SAVED] Fall detected → detected_falls/fall_20251113_165925_conf0.99.jpg
[SAVED] Fall detected → detected_falls/fall_20251113_165925_conf0.96.jpg
[SAVED] Fall detected → detected_falls/fall_20251113_165925_conf0.99.jpg
[SAVED] Fall detected → detected_falls/fall_20251113_165925_conf0.96.jpg
[SAVED] Fall detected → detected_falls/fall_20251113_165925_conf0.99.jpg
[SAVED] Fall detected → detected_falls/fall_20251113_165925_conf0.68.jpg
[SAVED] Fall detected → detected_falls/fall_20251113_165926_conf0.53.jpg
[SAVED] Fall detected → detected_falls/fall_20251113_165929_conf0.50.jpg
[SAVED] Fall detected → detected_falls/fall_20251113_165929_conf0.99.jpg
[SAVED] Fal